# Evaluate Customer Agent Conversations

Score multi-turn conversations for quality, context retention, query handling, loop detection, escalation, and prompt conformance — using built-in Turing metrics.

By the end of this notebook you will have scored a customer support conversation for overall quality, diagnosed specific failure modes like context loss and repetitive loops, checked whether the agent followed its system prompt, and run a full scorecard comparing a good conversation against a bad one.

> **Tip:** You can also pass a valid audio URL instead of a JSON conversation string for any of these metrics. Use `model_name="turing_large"` when evaluating audio inputs.

**Prerequisites:**
- FutureAGI account → [app.futureagi.com](https://app.futureagi.com)
- API keys: `FI_API_KEY` and `FI_SECRET_KEY` ([Get your API keys](https://docs.futureagi.com/admin-settings))
- Python 3.9+

## Install

In [ ]:
%pip install ai-evaluation --quiet

In [ ]:
import os

os.environ["FI_API_KEY"] = "your-api-key"        # Replace with your key
os.environ["FI_SECRET_KEY"] = "your-secret-key"  # Replace with your key

## Step 1: Set up the evaluator

In [ ]:
import os
import json
from fi.evals import Evaluator

evaluator = Evaluator(
    fi_api_key=os.environ["FI_API_KEY"],
    fi_secret_key=os.environ["FI_SECRET_KEY"],
)

## Step 2: Define two conversations

A well-handled support call and a poorly-handled one. Both start with the same customer issue — a duplicate charge.

In [ ]:
good_conversation = [
    {"role": "user", "content": "Hi, I was charged twice for order #7291. Can you help?"},
    {"role": "assistant", "content": "I'm sorry about that! Let me look up order #7291 right away."},
    {"role": "assistant", "content": "I can see the duplicate charge. I've initiated a refund of $49.99 to your original payment method. It should appear within 3-5 business days."},
    {"role": "user", "content": "Great, thank you. Will I get a confirmation email?"},
    {"role": "assistant", "content": "Yes, you'll receive a refund confirmation email within the next hour. Is there anything else I can help with?"},
    {"role": "user", "content": "No, that's all. Thanks for the quick help!"},
    {"role": "assistant", "content": "You're welcome! Don't hesitate to reach out if you need anything else. Have a great day!"},
]

bad_conversation = [
    {"role": "user", "content": "Hi, I was charged twice for order #7291. Can you help?"},
    {"role": "assistant", "content": "I'd be happy to help! Have you tried checking your account settings?"},
    {"role": "user", "content": "This isn't about account settings. I was charged $49.99 twice for the same order."},
    {"role": "assistant", "content": "I understand. Could you provide your order number?"},
    {"role": "user", "content": "I already said it's order #7291."},
    {"role": "assistant", "content": "Could you provide your order number so I can look into this?"},
    {"role": "user", "content": "I just told you twice — #7291. Can I speak to a manager?"},
    {"role": "assistant", "content": "I apologize for the inconvenience. Have you considered upgrading to our Premium support plan for faster resolution?"},
    {"role": "user", "content": "No! I just want my refund. Please transfer me to someone who can help."},
    {"role": "assistant", "content": "I understand your concern. Let me check your account settings to see if there's an issue."},
]

## Step 3: Score overall conversation quality

`customer_agent_conversation_quality` rates the overall interaction on a 1–5 scale considering clarity, helpfulness, responsiveness, and tone.

> **Note:** Choice-based metrics (quality, query handling, loop detection, escalation) return `eval_result.output` as a **list** (e.g., `['5']`). Score-based metrics (coherence, resolution, context retention) return a plain number. The snippets below use `output[0]` to extract the value when the output is a list.

In [ ]:
for label, convo in [("Good", good_conversation), ("Bad", bad_conversation)]:
    result = evaluator.evaluate(
        eval_templates="customer_agent_conversation_quality",
        inputs={"conversation": json.dumps(convo)},
        model_name="turing_small",
    )
    eval_result = result.eval_results[0]
    score = eval_result.output[0] if isinstance(eval_result.output, list) else eval_result.output
    print(f"{label} conversation: {score}/5")
    print(f"  Reason: {eval_result.reason}\n")

## Step 4: Diagnose why the bad conversation failed

Run targeted metrics on the bad conversation to pinpoint specific failure modes.

**Context retention** — did the agent remember details from earlier?

In [ ]:
result = evaluator.evaluate(
    eval_templates="customer_agent_context_retention",
    inputs={"conversation": json.dumps(bad_conversation)},
    model_name="turing_small",
)
eval_result = result.eval_results[0]
print(f"Context retention: {eval_result.output}")
print(f"Reason: {eval_result.reason}")

**Query handling** — did the agent correctly interpret and answer the user's questions?

In [ ]:
result = evaluator.evaluate(
    eval_templates="customer_agent_query_handling",
    inputs={"conversation": json.dumps(bad_conversation)},
    model_name="turing_small",
)
eval_result = result.eval_results[0]
score = eval_result.output[0] if isinstance(eval_result.output, list) else eval_result.output
print(f"Query handling: {score}")
print(f"Reason: {eval_result.reason}")

**Loop detection** — did the agent get stuck repeating the same prompts?

In [ ]:
result = evaluator.evaluate(
    eval_templates="customer_agent_loop_detection",
    inputs={"conversation": json.dumps(bad_conversation)},
    model_name="turing_small",
)
eval_result = result.eval_results[0]
score = eval_result.output[0] if isinstance(eval_result.output, list) else eval_result.output
print(f"Loop detection: {score}")
print(f"Reason: {eval_result.reason}")

**Human escalation** — did the agent escalate when the user asked for a manager?

In [ ]:
result = evaluator.evaluate(
    eval_templates="customer_agent_human_escalation",
    inputs={"conversation": json.dumps(bad_conversation)},
    model_name="turing_small",
)
eval_result = result.eval_results[0]
score = eval_result.output[0] if isinstance(eval_result.output, list) else eval_result.output
print(f"Human escalation: {score}")
print(f"Reason: {eval_result.reason}")

## Step 5: Evaluate prompt conformance

`customer_agent_prompt_conformance` checks whether the agent followed its system prompt throughout the conversation. This is the only conversation metric that takes an additional `system_prompt` input.

In [ ]:
system_prompt = (
    "You are a billing support agent for TechStore. "
    "Your role is to help customers resolve payment and billing issues. "
    "Always acknowledge the customer's issue first, then investigate. "
    "Never upsell products during a support interaction. "
    "If a customer asks to speak with a manager, escalate immediately."
)

for label, convo in [("Good", good_conversation), ("Bad", bad_conversation)]:
    result = evaluator.evaluate(
        eval_templates="customer_agent_prompt_conformance",
        inputs={
            "system_prompt": system_prompt,
            "conversation": json.dumps(convo),
        },
        model_name="turing_small",
    )
    eval_result = result.eval_results[0]
    score = eval_result.output[0] if isinstance(eval_result.output, list) else eval_result.output
    print(f"{label} conversation — prompt conformance: {score}")
    print(f"  Reason: {eval_result.reason}\n")

## Step 6: Full scorecard

Run all key metrics on both conversations in a single diagnostic sweep.

In [ ]:
metrics = [
    ("conversation_coherence", "Coherence"),
    ("conversation_resolution", "Resolution"),
    ("customer_agent_conversation_quality", "Quality"),
    ("customer_agent_context_retention", "Context"),
    ("customer_agent_query_handling", "Queries"),
    ("customer_agent_loop_detection", "Loops"),
    ("customer_agent_human_escalation", "Escalation"),
]

print(f"{'Metric':<14}  {'Good':>12}  {'Bad':>12}")
print("-" * 42)

for metric_name, label in metrics:
    good_result = evaluator.evaluate(
        eval_templates=metric_name,
        inputs={"conversation": json.dumps(good_conversation)},
        model_name="turing_small",
    )
    bad_result = evaluator.evaluate(
        eval_templates=metric_name,
        inputs={"conversation": json.dumps(bad_conversation)},
        model_name="turing_small",
    )
    good_raw = good_result.eval_results[0].output
    bad_raw = bad_result.eval_results[0].output
    good_val = good_raw[0] if isinstance(good_raw, list) else good_raw
    bad_val = bad_raw[0] if isinstance(bad_raw, list) else bad_raw
    print(f"{label:<14}  {str(good_val):>12}  {str(bad_val):>12}")

## What you built

- Scored a customer support conversation for overall quality with `customer_agent_conversation_quality`
- Diagnosed specific failure modes: context loss, poor query handling, repetitive loops, and missed escalation
- Checked whether the agent followed its system prompt with `customer_agent_prompt_conformance`
- Ran a full scorecard comparing a good conversation against a bad one across 7 metrics

### Next steps

- [Chat Simulation with Personas](https://docs.futureagi.com/cookbook/quickstart/chat-simulation-personas) — generate multi-turn conversations and evaluate at scale
- [Session Observability](https://docs.futureagi.com/cookbook/quickstart/session-observability) — track multi-turn conversations in production
- [Batch Evaluation](https://docs.futureagi.com/cookbook/quickstart/batch-eval) — run conversation evals across dataset rows
- [All Built-in Metrics](https://docs.futureagi.com/future-agi/get-started/evaluation/builtin-evals/overview) — 72+ built-in eval metrics